# 02 · Limpieza, segmentación y aristas — IVR Alkosto

Toma `df_pasos` (salida de `01_Alk_traza.ipynb`, ya en formato largo y
en el orden cronológico correcto, sin necesidad de volver a hacer
`split('|')`/`split(';')`) y hace lo que hacía la segunda mitad de
`01_Alk_final.ipynb`: cruzar contra el maestro, asignar Nodo, clasificar cada
paso, y construir la arista origen→destino (`op_text_final`) con `shift(-1)`.

#### Decisión clave: cruzar por `Llave`, no solo por `CodigoTraza`

Al inspeccionar `Maestro_trazas_alkosto.xlsx` (hoja `Detalle MaestroTrazasOpciones`)
encontramos que **`CodigoTraza` no es único** — hay 5 códigos que se repiten con un
texto distinto (p. ej. código `5` = `Horario de nuestras tiendas` **y**
`Menu horario tiendas`; código `327` = dos variantes de `¿Estado devolución en
punto de venta?` con/sin tilde). En cambio **`Llave` (`CodigoTraza_TrazaOpcion`) sí
es 100% única** (142/142 filas). Por eso cruzamos por `Llave`, construida igual en
nuestros datos (`op_num` + `_` + `op_text` recortado), y dejamos el cruce por texto
solo como respaldo (normalizando tildes con `unidecode`) para los casos que no
matcheen por `Llave` exacta.

#### Decisión clave: los `Nodo 1..10` ahora son una ruta jerárquica, no una categoría

En el maestro original, `Nodo N` eran categorías sueltas y el pipeline viejo
buscaba en qué columna aparecía el texto (el propio README dice que ese paso
"fue innecesario"). En el maestro actual, cada fila trae el **camino completo**
desde la raíz del árbol hasta esa opción (`Nodo 1` = `Inicio IVR`, `Nodo 2` =
`Menú Principal`, `Nodo 3` = `Horario de nuestras tiendas`, etc.). Conservamos las
10 columnas tal cual (son un breadcrumb útil para Power BI) y agregamos
`nodo_final` = el último nodo no nulo de esa ruta, como resumen rápido.

#### Clasificación (reemplazo de `Segmentacion`)

`Clasifica Traza` en el maestro actual **solo trae `'Navegación'`** en las 142
filas — no distingue nada. La distinción real de "tipo de gestión" vive en
`Efectivo` (Sí/No) + `Clasifica Efectivo` (`Horarios`, `Garantias`, `Despachos`,
`Seguros celulares`, `Tarjeta Alkosto`, `Credito 20 minutos`). Verificamos además
que los 3 "Paso agente..." que ya existen en el maestro (`despachos`, `garantias`,
`instalaciones`) están marcados `Efectivo=No` — el maestro **no** los distingue
como "paso a asesor", así que seguimos necesitando la regla manual por texto
("paso" en `op_text`), igual que en `01_Alk_final.ipynb`.

Construimos entonces:
- `segmentacion`: `'Paso asesor'` (regla de texto) > `'Navegación'` (si matcheó en
  el maestro) > `'no identificado'` (si no matcheó por ningún método).
- `efectivo` / `clasifica_efectivo`: se conservan tal cual del maestro, para los
  filtros por escenario del Notebook 4 (Garantías, Despachos, Horarios...).

**Entrada:** `df_pasos_<periodo>.parquet` (Notebook 2) +
`Maestro_trazas_alkosto.xlsx`.

**Salida:** `df_transformado_<periodo>.parquet` — equivalente al
`df_transformado.xlsx` del pipeline original, con `op_text_final` ya construido.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
from unidecode import unidecode

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

### Parámetros — deben coincidir con los notebooks anteriores

In [ ]:
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
STAGE_DIR = DATA_DIR / "01_staging"
PROC_DIR = DATA_DIR / "02_procesado"
PROC_DIR.mkdir(parents=True, exist_ok=True)

MAESTRO_PATH = Path("C:/Users/jupasoro/OneDrive - Emtelco/Proyectos/Alkosto/grafos_ivr_2026/Maestro trazas alkosto.xlsx")

IN_PASOS_PATH = STAGE_DIR / f"df_pasos_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_TRANSFORMADO_PATH = PROC_DIR / f"df_transformado_{FECHA_INICIO}_{FECHA_FIN}.parquet"

assert IN_PASOS_PATH.exists(), f"No encuentro {IN_PASOS_PATH} — corre primero 02_reconstruccion_traza.ipynb"
assert MAESTRO_PATH.exists(), f"No encuentro {MAESTRO_PATH} — ajusta MAESTRO_PATH"
print(f"Leyendo pasos: {IN_PASOS_PATH}")
print(f"Leyendo maestro: {MAESTRO_PATH}")

Leyendo pasos: data\01_staging\df_pasos_2026-06-01_2026-06-30.parquet
Leyendo maestro: C:\Users\jupasoro\OneDrive - Emtelco\Proyectos\Alkosto\grafos_ivr_2026\Maestro trazas alkosto.xlsx


In [ ]:
df_pasos = pd.read_parquet(IN_PASOS_PATH)
print("df_pasos:", df_pasos.shape)
df_pasos.head(3)

df_pasos: (1296067, 6)


,id_conversacion,orden,op_num,op_text,op_tiempo,origen_campo
0,891f883b-e8c0-4a43-91a8-00f93459a745,0,0,Inicio IVR,0,traza_opciones
1,891f883b-e8c0-4a43-91a8-00f93459a745,1,2,Numero documento ingresado,1023941473,custom2
2,891f883b-e8c0-4a43-91a8-00f93459a745,2,3,Habeas data positivo,48030,traza_opciones


#### Cargar y preparar el maestro

In [ ]:
maestro = pd.read_excel(MAESTRO_PATH, sheet_name="Detalle MaestroTrazasOpciones")

columnas_nodo = [f"Nodo {i}" for i in range(1, 11)]
columnas_maestro_utiles = [
    "Llave",
    "CodigoTraza",
    "TrazaOpcion",
    "Efectivo",
    "Clasifica Efectivo",
    "Clasifica Traza",
    "TrazaFallaWebServ",
    "VoiceBot",
    "Clasifica Voicebot",
    "Nombre IVR",
    "CodigoOrden",
] + columnas_nodo

maestro = maestro[columnas_maestro_utiles].copy()
maestro["TrazaOpcion"] = maestro["TrazaOpcion"].astype(str).str.strip()

print("Filas en maestro:", len(maestro))
print("Llave únicas:", maestro["Llave"].nunique(), "/", len(maestro))
assert maestro["Llave"].is_unique, "Llave dejó de ser única — revisa el maestro actualizado"
maestro.head(3)

Filas en maestro: 142
Llave únicas: 142 / 142


,Llave,CodigoTraza,TrazaOpcion,Efectivo,Clasifica Efectivo,Clasifica Traza,TrazaFallaWebServ,VoiceBot,Clasifica Voicebot,Nombre IVR,CodigoOrden,Nodo 1,Nodo 2,Nodo 3,Nodo 4,Nodo 5,Nodo 6,Nodo 7,Nodo 8,Nodo 9,Nodo 10
0,0_Inicio IVR,0,Inicio IVR,No,NaN,Navegación,No,No,NaN,IVR General Alkosto,11000000000,Inicio IVR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1_Confirmar documento,1,Confirmar documento,No,NaN,Navegación,No,No,NaN,IVR General Alkosto,11100000000,Inicio IVR,Confirmar documento,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2_Modificar documento,2,Modificar documento,No,NaN,Navegación,No,No,NaN,IVR General Alkosto,11200000000,Inicio IVR,Modificar documento,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Construir `llave` en `df_pasos` con el mismo formato del maestro (`CodigoTraza_TrazaOpcion`)

In [ ]:
df_pasos["op_text"] = df_pasos["op_text"].astype(str).str.strip()
df_pasos["op_num_int"] = pd.to_numeric(df_pasos["op_num"], errors="coerce")

codigo_no_numerico = df_pasos["op_num_int"].isna().sum()
if codigo_no_numerico:
    print(f"Aviso: {codigo_no_numerico} pasos con op_num no numérico — no podrán armar 'llave', quedarán 'no identificado'.")

df_pasos["llave"] = (
    df_pasos["op_num_int"].astype("Int64").astype(str) + "_" + df_pasos["op_text"]
)
df_pasos.loc[df_pasos["op_num_int"].isna(), "llave"] = pd.NA

#### Cruce principal por `llave` (exacto: código + texto)

In [ ]:
df_join = df_pasos.merge(
    maestro.add_suffix("_maestro").rename(columns={"Llave_maestro": "llave"}),
    on="llave",
    how="left",
)

matcheo_llave = df_join["CodigoTraza_maestro"].notna().mean()
print(f"% de pasos matcheados por llave exacta: {matcheo_llave:.1%}")

% de pasos matcheados por llave exacta: 33.0%


#### Cruce de respaldo por texto normalizado (sin tildes/mayúsculas)

Para los que no matchearon por `llave` exacta — cubre casos como el par
`Estado devolución`/`Estado devolucion` que vimos en el maestro (mismo código,
texto con y sin tilde).

In [ ]:
def normalizar(texto):
    return unidecode(str(texto)).strip().lower()

maestro["texto_normalizado"] = maestro["TrazaOpcion"].apply(normalizar)
# Si el texto normalizado se repite (varias filas del maestro), nos quedamos con
# la primera ocurrencia — es un respaldo de baja precisión, no la vía principal.
lookup_texto = (
    maestro.drop_duplicates(subset="texto_normalizado", keep="first")
    .set_index("texto_normalizado")
)

sin_match = df_join["CodigoTraza_maestro"].isna()
df_join.loc[sin_match, "texto_normalizado"] = df_join.loc[sin_match, "op_text"].apply(normalizar)

columnas_a_rellenar = [c for c in df_join.columns if c.endswith("_maestro")]
for col in columnas_a_rellenar:
    col_original = col.replace("_maestro", "")
    if col_original == "texto_normalizado" or col_original not in lookup_texto.columns:
        continue
    valores_respaldo = df_join.loc[sin_match, "texto_normalizado"].map(lookup_texto[col_original])
    df_join.loc[sin_match, col] = df_join.loc[sin_match, col].fillna(valores_respaldo)

df_join["matched_por"] = "llave"
df_join.loc[sin_match & df_join["CodigoTraza_maestro"].notna(), "matched_por"] = "texto_normalizado"
df_join.loc[df_join["CodigoTraza_maestro"].isna(), "matched_por"] = "sin_match"

print(df_join["matched_por"].value_counts())
print(f"\n% total matcheado (llave + respaldo texto): {(df_join['matched_por'] != 'sin_match').mean():.1%}")

matched_por
sin_match            852966
llave                427521
texto_normalizado     15580
Name: count, dtype: int64

% total matcheado (llave + respaldo texto): 34.2%


#### Revisión de los `op_text` que quedaron sin match

Antes de seguir, revisa esta lista contra `Flujo_actualizado_Alkosto_corte_28-04-26.pdf`
— puede haber opciones nuevas del IVR que el maestro todavía no tiene registradas.

In [ ]:
no_identificados = (
    df_join[df_join["matched_por"] == "sin_match"]
    .groupby("op_text")
    .size()
    .sort_values(ascending=False)
    .rename("n_ocurrencias")
    .reset_index()
)
print(f"{len(no_identificados)} textos distintos sin identificar ({no_identificados['n_ocurrencias'].sum()} pasos en total)")
no_identificados.head(30)

85 textos distintos sin identificar (852966 pasos en total)


,op_text,n_ocurrencias
0,Consulta ws GetClientByAni,82069
1,Usuario_Identificado_Con_ANI,82010
2,Consulta ws CheckAftersalesCases,46034
3,Numero documento ingresado,45699
4,Existe habeas data,35366
5,Consulta ws GetClientByDocument,34769
6,No input,31514
7,Usuario_Identificado_Con_Ani,31034
8,Primera pregunta SAC,29964
9,Bienvenida encuesta SAC,29964


#### Clasificación final (`segmentacion`) y `nodo_final`

In [ ]:
# Regla manual de respaldo: cualquier op_text que contenga "paso" se marca como
# Paso asesor, igual que en 01_Alk_final.ipynb — el maestro clasifica los 3
# "Paso agente..." que ya conoce como Efectivo=No / Navegación, no como un tipo
# aparte, así que esta regla sigue siendo necesaria.
es_paso_asesor = df_join["op_text"].str.lower().str.contains("paso", na=False)

df_join["segmentacion"] = "no identificado"
df_join.loc[df_join["matched_por"] != "sin_match", "segmentacion"] = "Navegación"
df_join.loc[es_paso_asesor, "segmentacion"] = "Paso asesor"

print(df_join["segmentacion"].value_counts())
print("\nEfectivo (viene del maestro, Sí/No/nulo si no matcheó):")
print(df_join["Efectivo_maestro"].value_counts(dropna=False))
print("\nClasifica Efectivo:")
print(df_join["Clasifica Efectivo_maestro"].value_counts(dropna=False))

segmentacion
no identificado    829176
Navegación         413871
Paso asesor         53020
Name: count, dtype: int64

Efectivo (viene del maestro, Sí/No/nulo si no matcheó):
Efectivo_maestro
NaN    852966
No     434377
Si       8724
Name: count, dtype: int64

Clasifica Efectivo:
Clasifica Efectivo_maestro
NaN                   1287343
Despachos                4432
Tarjeta Alkosto          3304
Horarios                  593
Credito 20 minutos        395
Name: count, dtype: int64


In [ ]:
columnas_nodo_maestro = [f"Nodo {i}_maestro" for i in range(1, 11)]


def ultimo_nodo_no_nulo(fila):
    for col in reversed(columnas_nodo_maestro):
        valor = fila[col]
        if pd.notna(valor) and str(valor).strip().lower() not in ("", "null", "none"):
            return valor
    return None


df_join["nodo_final"] = df_join[columnas_nodo_maestro].apply(ultimo_nodo_no_nulo, axis=1)
df_join["nodo_final"] = df_join["nodo_final"].fillna("no identificado")
df_join["nodo_final"].value_counts().head(15)

nodo_final
no identificado                               852966
Inicio IVR                                     82545
Menú Principal                                 78702
Garantias y devoluciones                       21675
¿Estado devolución en punto de venta? = NO     17090
Estado de entrega                              17041
¿Factura encontrada? = ERROR SI                15936
Habeas data positivo                           15900
Paso agente despachos                          14232
¿Total de facturas mayor a uno? = NO           13543
Informacion general                            13467
Validación factura correcta                    11694
Paso agente garantias                          10664
¿Factura igual a DES? = SI                     10338
¿Factura encontrada? = ERROR NO                 9891
Name: count, dtype: int64

#### Construcción de la arista origen→destino (`op_text_final`)

Igual que en `01_Alk_final.ipynb`: `op_text_final` es `op_text` desplazado una
posición hacia abajo dentro de cada `id_conversacion` (usando el orden ya
correcto de `orden`, resultado del Notebook 2). La última fila de cada
conversación se marca como `final`.

In [ ]:
df_join = df_join.sort_values(["id_conversacion", "orden"]).reset_index(drop=True)

df_join["op_text_final"] = df_join.groupby("id_conversacion")["op_text"].shift(-1)
ultimas_filas = df_join.groupby("id_conversacion").tail(1).index
df_join.loc[ultimas_filas, "op_text_final"] = "final"

# Recontamos 'orden' para que quede 0..n-1 sin huecos (por si algún paso se
# hubiera descartado en el camino), igual que 01_Alk_final.ipynb hacía con 'bloque'.
df_join["orden"] = df_join.groupby("id_conversacion").cumcount()

df_join[["id_conversacion", "orden", "op_text", "op_text_final", "segmentacion", "nodo_final", "origen_campo"]].head(10)

,id_conversacion,orden,op_text,op_text_final,segmentacion,nodo_final,origen_campo
0,0000b8f3-39e8-4388-aa08-23c0031a9d9f,0,Inicio IVR,Numero documento ingresado,Navegación,Inicio IVR,traza_opciones
1,0000b8f3-39e8-4388-aa08-23c0031a9d9f,1,Numero documento ingresado,Habeas data negativo,no identificado,no identificado,custom2
2,0000b8f3-39e8-4388-aa08-23c0031a9d9f,2,Habeas data negativo,Existe habeas data,Navegación,Habeas data negativo,traza_opciones
3,0000b8f3-39e8-4388-aa08-23c0031a9d9f,3,Existe habeas data,Agendar servicio de instalación,no identificado,no identificado,custom11
4,0000b8f3-39e8-4388-aa08-23c0031a9d9f,4,Agendar servicio de instalación,Usuario_Identificado_Con_ANI,Navegación,Agendar servicio de instalación,traza_opciones
5,0000b8f3-39e8-4388-aa08-23c0031a9d9f,5,Usuario_Identificado_Con_ANI,Menu principal,no identificado,no identificado,custom16
6,0000b8f3-39e8-4388-aa08-23c0031a9d9f,6,Menu principal,Consulta ws ConsultaHabeasData,Navegación,Menú Principal,traza_opciones
7,0000b8f3-39e8-4388-aa08-23c0031a9d9f,7,Consulta ws ConsultaHabeasData,Consulta ws ActualizaHabeasData,no identificado,no identificado,custom22
8,0000b8f3-39e8-4388-aa08-23c0031a9d9f,8,Consulta ws ActualizaHabeasData,Consulta ws CheckAftersalesCases,no identificado,no identificado,custom21
9,0000b8f3-39e8-4388-aa08-23c0031a9d9f,9,Consulta ws CheckAftersalesCases,Consulta ws GetClientByAni,no identificado,no identificado,custom28


#### Selección final de columnas y exportación

In [ ]:
df_transformado = df_join.rename(
    columns={
        "Efectivo_maestro": "efectivo",
        "Clasifica Efectivo_maestro": "clasifica_efectivo",
        "CodigoTraza_maestro": "codigo_traza_maestro",
        "TrazaOpcion_maestro": "traza_opcion_maestro",
        "Nombre IVR_maestro": "nombre_ivr",
        "CodigoOrden_maestro": "codigo_orden",
    }
)[
    [
        "id_conversacion",
        "orden",
        "op_num",
        "op_text",
        "op_tiempo",
        "op_text_final",
        "origen_campo",
        "matched_por",
        "segmentacion",
        "efectivo",
        "clasifica_efectivo",
        "nodo_final",
        "nombre_ivr",
        "codigo_orden",
    ]
    + columnas_nodo_maestro
]

df_transformado.to_parquet(OUT_TRANSFORMADO_PATH, index=False)
print(f"Guardado: {OUT_TRANSFORMADO_PATH}  ({len(df_transformado)} filas)")

muestra_path = PROC_DIR / f"df_transformado_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_transformado.head(500).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

Guardado: data\02_procesado\df_transformado_2026-06-01_2026-06-30.parquet  (1296067 filas)
Muestra Excel: data\02_procesado\df_transformado_muestra_2026-06-01_2026-06-30.xlsx


---
**Antes de seguir al Notebook 4 (filtrado por escenario y grafos):**
1. Revisa el % de `matched_por == 'sin_match'` — si es alto, hay que actualizar
   el maestro o mapear manualmente esos textos contra el PDF del flujo antes de
   continuar (una clasificación `no identificado` alta ensucia el grafo).
2. Revisa la lista de `no_identificados` impresa arriba contra el flujo vigente.

**Siguiente paso:** `04_escenarios.ipynb` — usa la hoja `Ultima traza` del maestro
(distinta de `Detalle MaestroTrazasOpciones`) para identificar el escenario final
de cada `id_conversacion` (equivalente al viejo `traza_final`), filtra
`df_transformado` por escenario (menú, garantías, entregas, transferencia a Tuya,
existencia de producto) y por cantidad de pasos, y exporta los `base_*.xlsx` que
consume el Notebook 5 (grafos).